# 02 · From observations to a comparable signal

## Context

Climate observations are often exchanged as one row per date and location.
Here we flatten official PRISM minimum temperature into that familiar table,
then recover the cube without changing a value.

## Question

Can we compare departures through time at sites with different baseline
temperatures?

## Analysis story

We will prove the table-to-cube round trip, standardize each location through
time, and compare the observed and standardized series at one grid cell.


### Data used in this lesson

Every value comes from the PRISM Group at Oregon State University's AN91d
daily 4 km climate product. This repository carries a small Boulder-region
extract for 1–30 January 2024 so the lesson runs offline without replacing
observations with generated values. The [data validation page](../validation/data.md)
records source URLs, terms, checksums, bounds, units, and acceptance tests.

In [ ]:
# Record the exact code imported by this notebook kernel.
import cubedynamics as cd

print(cd.version_info())


## Prepare · Round-trip real observations through a tidy table

In [ ]:
from pathlib import Path

import xarray as xr

# Find the repository from either a root-level documentation build or a kernel
# started beside this notebook, then open the checksum-controlled PRISM extract.
data_path = next(
    candidate / "tests" / "fixtures" / "real_data" / "prism_boulder_january_2024.nc"
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "tests" / "fixtures" / "real_data" / "prism_boulder_january_2024.nc").exists()
)
prism = xr.open_dataset(data_path, engine="scipy").load()

# These assertions are part of the teaching contract: official source,
# canonical cube dimensions, complete daily time, and declared Celsius units.
assert prism.attrs["source"] == "PRISM Group, Oregon State University"
assert prism.attrs["is_synthetic"] == 0
assert prism.sizes == {"time": 30, "y": 24, "x": 24}
assert prism["tmax"].attrs["units"] == "degC"

import numpy as np

source = prism["tmin"].isel(y=slice(9, 14), x=slice(9, 15))
table = source.to_dataframe(name="tmin").reset_index()

cube = (
    table.set_index(["time", "y", "x"])
    .to_xarray()["tmin"]
    .transpose("time", "y", "x")
    .sel(time=source.time, y=source.y, x=source.x)
)
cube.attrs.update(source.attrs)
cube.attrs.update(source=prism.attrs["source"], is_synthetic=0)

np.testing.assert_array_equal(cube.values, source.values)
assert len(table) == source.size
cube

## Pipe · Standardize each location through time

Preparation is complete. The analytical method is one verb.

In [ ]:
from cubedynamics import pipe, verbs as v

standardized = (
    pipe(cube)
    | v.zscore(dim="time")
).unwrap()

assert float(abs(standardized.mean("time")).max()) < 1e-5
standardized

## Figure · Compare the observed and standardized records

In [ ]:
import matplotlib.pyplot as plt

site = {"y": float(cube.y[2]), "x": float(cube.x[3])}
fig, axes = plt.subplots(2, 1, figsize=(9, 5.5), sharex=True, constrained_layout=True)
cube.sel(**site).plot(ax=axes[0], marker="o", color="#8b543c")
axes[0].set_title("Observed PRISM daily minimum temperature")
axes[0].set_ylabel("Temperature (°C)")
standardized.sel(**site).plot(ax=axes[1], marker="o", color="#3f6f72")
axes[1].axhline(0, color="0.35", linewidth=0.8)
axes[1].set_title("The same grid cell after v.zscore(dim='time')")
axes[1].set_ylabel("Standard deviations")
plt.show()

## What the figure tells us

Standardization preserves the January timing—including the sharp cold
outbreak—while expressing each value relative to that location's own record.

## Try the next variation

Replace `v.zscore` with `v.anomaly` and explain which scale is more useful for
your question.

## Data used

| Field | Frozen analysis input |
| --- | --- |
| Provider | PRISM Group, Oregon State University |
| Product | AN91d daily 4 km time series |
| Dates | 2024-01-01 to 2024-01-30 |
| Fixture | `tests/fixtures/real_data/prism_boulder_january_2024.nc` |
| Provenance record | `tests/fixtures/real_data/prism_boulder_january_2024.provenance.json` |

The [PRISM source reference](../library/sources/prism.md) describes current
catalog support; the fixture record above identifies the observations used
here. [Data validation](../validation/data.md) documents checksums and acceptance
checks. The analytical baseline and thresholds belong to this story, not the provider.

## Reproduce

Clone the repository, then run these commands from its root:

```bash
python -m pip install -e ".[vignettes]"
python scripts/run_vignettes.py docs/vignettes/cube_from_tidy_table.ipynb
```

No network is needed after installation. Open the downloaded notebook in
Jupyter and run all cells to see the same figures. The website executes these
cells during its strict build. [Environment setup](../learn/index.md#shared-setup)
and the [vignette contract](../vignettes/structure.md) explain the workflow.
The first code cell prints `cd.version_info()` so a rendered result can be tied
to a package path and, for development checkouts, a Git commit.

## See also

[temperature](../library/nouns/temperature.md) · [precipitation](../library/nouns/precipitation.md) ·
[zscore](../reference/verbs/zscore.md)

[Learn the grammar](../learn/index.md) · [All vignettes](../vignettes/index.md)
